# Session 1: Build a scene prompt

Describe a scene with a small dictionary, turn it into complete sentences, then generate and save an image.

[Open in Colab](https://colab.research.google.com/github/digitalideation/prompt_h2601/blob/simplify-replicate-basics/notebooks/02_replicate_scene_prompts.ipynb), or open this notebook in JupyterHub. You only need this notebook file. The API steps are the same as in notebook 01.

Use your own Replicate account with API access and credit. Run the cells in order with **Shift+Enter**. Generation runs on Replicate, so you do not need a GPU in your notebook environment.

## 1. Install Replicate

Run this once when you start a new environment. It installs the Python client used to send requests to Replicate.

In [ ]:
%pip install -q replicate

## 2. Connect your account

Copy your token from [Replicate's API token page](https://replicate.com/account/api-tokens) and paste it into the hidden input below. Never put the token in a code cell or share it. This step does not generate an image.

In [ ]:
import replicate
from getpass import getpass

client = replicate.Client(api_token=getpass("Replicate API token: ").strip(), timeout=120)

## 3a. Describe your scene

Edit the text values in the dictionary. Think **Who? What? Where? When? How?**

- **Subject:** a person, object or group, such as "a cyclist" or "two dancers".
- **Action:** begin with an activity such as "riding", "standing" or "walking".
- **Setting and time:** a place that fits after "in", and a time that fits after "at".
- **Style, lighting, composition, palette and mood:** describe the visual treatment.

Use short phrases without final full stops. These field names organise your description; they are not separate Replicate settings.

In [ ]:
scene = {
    "subject": "a cyclist wearing a mustard-yellow raincoat",
    "action": "riding slowly across a stone bridge",
    "setting": "a quiet riverside town",
    "time": "dawn",
    "lighting": "warm orange light filtering through mist",
    "style": "cinematic photography",
    "composition": "a wide shot from street level",
    "palette": "muted blue-grey tones",
    "mood": "quiet and contemplative",
}

## 3b. Build and read your prompt

This template connects your phrases into sentences. In Python, an `f` before a quoted string lets you insert values inside `{...}`.

Run the cell and read the complete prompt. Check that the action, place, lighting and mood make sense together. You can edit the connecting words in the template if your scene needs different wording.

This step only assembles text; it makes no API request. After editing the dictionary, rerun both **3a and 3b** before generating.

In [ ]:
prompt = (
    f"Show {scene['subject']} {scene['action']}. "
    f"The scene takes place in {scene['setting']} at {scene['time']}, "
    f"with {scene['lighting']}. "
    f"Use {scene['style']} and frame the scene as {scene['composition']}. "
    f"Use {scene['palette']} throughout the image. "
    f"The overall mood should feel {scene['mood']}."
)
print(prompt)

## 4. Generate one image

We use [FLUX.1 schnell](https://replicate.com/black-forest-labs/flux-schnell). Run the function cell once: it looks up the model version, creates a prediction, prints its ID and waits for the image URL.

Then run the cell below it to generate one square PNG. **Each run of the generation cell requests a new image and may incur a charge.** If a request fails or you interrupt it, check the printed prediction ID in Replicate before trying again.

In [ ]:
class GenerationError(Exception):
    """A prediction finished without a usable image."""


def generate_image(inputs, client, MODEL):
    model = client.models.get(MODEL)
    if model.latest_version is None:
        raise GenerationError("This model has no available version. Check the model with the instructor.")
    version_id = model.latest_version.id

    prediction = client.predictions.create(version=version_id, input=inputs)
    print("Prediction ID:", prediction.id)
    prediction.wait()

    if prediction.status != "succeeded":
        raise GenerationError(
            f"Prediction {prediction.id} ended with status '{prediction.status}'. "
            "Check its details in Replicate before trying again."
        )
    urls = prediction.output
    if not isinstance(urls, list) or not urls or not isinstance(urls[0], str) or not urls[0].startswith("https://"):
        raise GenerationError(
            f"Prediction {prediction.id} returned no usable image URL. "
            "Check the model's output format with the instructor."
        )
    return urls

In [ ]:
output = None
image_bytes = None
MODEL = "black-forest-labs/flux-schnell"
inputs = {
    "prompt": prompt,
    "num_outputs": 1,
    "aspect_ratio": "1:1",
    "output_format": "png",
}
output = generate_image(inputs, client, MODEL)

## 5. View the result

The function returns image URLs. This cell downloads the first image and displays it. Running this cell again does not generate another image. If downloading fails, retry this cell.

In [ ]:
from urllib.request import urlopen
from IPython.display import Image, display

with urlopen(output[0], timeout=120) as response:
    image_bytes = response.read()
display(Image(data=image_bytes, width=600))

## 6. Save the image (optional)

Choose a filename ending in `.png`. Use a new name for each image you want to keep; an existing file will not be replaced.

The file is saved in your notebook environment. Download it through the file browser: the **Files** sidebar in Colab, or the file browser in JupyterHub. Download Colab files before ending your session.

In [ ]:
filename = "my-scene-01.png"

if image_bytes is None:
    print("Run step 5 to load the image first.")
else:
    with open(filename, "xb") as file:
        file.write(image_bytes)
    print("Saved:", filename)

## Try one change

Save any result you want to keep. Change one field in the dictionary, for example `composition` to "a wide shot from above". Rerun **3a and 3b**, read the new prompt, then run the generation cell in **4** and view the result in **5**.

Use a different filename in step **6** if you save the new image. Keep your notebook copy to retain the scene description and prompt.

Did the viewpoint change as intended? What else changed? A single pair of images is an observation, not proof that a prompt always works better.

When you finish, restart the kernel or runtime to clear your API connection.